# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This section lists the `@id`s and fields of each record set in the dataset. Each `@id` uniquely identifies an entity in the Croissant schema.

In [ ]:
# Get all record set @ids
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- Record set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}")
        print(f"      Name: {field.name}")
        print(f"      Data type: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. The `dataset.records(record_set=...)` API uses the record set's `@id`.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Retrieve records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nColumns for record set {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())
else:
    print("No record sets available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and categorizing data.

**Note:** In this example, we analyze one numeric field from the first record set, referencing by its `@id`.

In [ ]:
import numpy as np

# Select record set and a numeric field for analysis
# We'll use the first record set and try to find a numeric field
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    rs = [rs for rs in dataset.record_sets if rs.id == record_set_id][0]
    # Search for a numeric field
    numeric_field = None
    for field in rs.fields:
        if (isinstance(field.data_type, str) and ("int" in field.data_type.lower() or "float" in field.data_type.lower() or "number" in field.data_type.lower())) and (field.id in df.columns):
            numeric_field = field.id
            break
    if numeric_field is None and len(df.select_dtypes(include=[np.number]).columns) > 0:
        numeric_field = df.select_dtypes(include=[np.number]).columns[0]

    print(f"Using record set: {record_set_id}")
    print(f"Numeric field selected (by @id): {numeric_field}")

    threshold = 10
    if numeric_field and numeric_field in df.columns:
        # Drop NA for safe filtering
        filtered_df = df[df[numeric_field].fillna(0) > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the values
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by first available categorical field
        group_field = None
        for field in rs.fields:
            if (field.data_type in ["schema:Text", "schema:Boolean"] or "str" in str(field.data_type).lower()) and field.id in df.columns and field.id != numeric_field:
                group_field = field.id
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in record set.")
else:
    print("No record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn. The following example plots the distribution of a numeric field and a scatter plot against a categorical/group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library. We loaded dataset metadata, listed record set and field `@id`s, loaded the records into pandas DataFrames, and conducted basic exploratory data analysis and visualization on one record set. 

To further analyze this dataset, refer to the Croissant schema documentation for richer insights on all available fields and data relationships.